In [1]:
import os
import time
import json

import openai
from openai import OpenAI

from elasticsearch import Elasticsearch

import spacy
import numpy as np

import json
import pandas as pd
from tqdm.auto import tqdm

In [2]:
es_client = Elasticsearch('http://localhost:9200') 

index_name='documents'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents: 79


In [3]:
DEFAULT_API_KEY = os.environ.get("OPENAI_API_KEY")

nlp = spacy.load('en_core_web_sm')

In [21]:
def get_vector(query):
    doc = nlp(query)
    tokens = [token.lemma_ for token in doc]
    text = ' '.join(tokens)
    doc_lemmatized = nlp(text)
    vector = np.mean([token.vector for token in doc_lemmatized], axis=0).tolist()
    return vector

In [22]:
query = "How can i create my account?"
query

'How can i create my account?'

In [23]:
vector = get_vector(query)
len(vector)

96

In [25]:

keyword_query = {
    "bool": {
        "must": {
            "multi_match": {
                "query": query,
                "fields": ["question^3", "answer"],
                "type": "best_fields",
                "boost": 0.5,
            }   
        },
    }
}

In [26]:
knn_query = {
    "field": "embedding",
    "query_vector": vector,
    "k": 5,
    "num_candidates": 10000,
    "boost": 0.5,
}

In [27]:
search_query_knn = {
    "knn": {
        "field": "embedding",
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000,
    },
    "size": 5,
    "_source": ['document_id', 'question', 'answer'],
}

es_results = es_client.search(index=index_name, body=search_query_knn)

[elem['_score'] for elem in es_results["hits"]["hits"]]

[0.6908349, 0.6800631, 0.6793462, 0.6775595, 0.67575777]

In [28]:
[hit["_source"] for hit in es_results["hits"]["hits"]]

[{'document_id': 'doc_8_can_i_change_my_shipping_addre',
  'question': 'Can I change my shipping address after placing an order?',
  'answer': 'If you need to change your shipping address, please contact our customer support team as soon as possible. We will do our best to update the address if the order has not been shipped yet.'},
 {'document_id': 'doc_21_what_should_i_do_if_i_receive_',
  'question': 'What should I do if I receive the wrong item?',
  'answer': 'If you receive the wrong item in your order, please contact our customer support team immediately. We will arrange for the correct item to be shipped to you and assist with returning the wrong item.'},
 {'document_id': 'doc_64_can_i_request_a_product_if_it_',
  'question': 'Can I request a product if it is not listed on your website?',
  'answer': 'If a product is not listed on our website, it may not be available for purchase. We recommend exploring the available products or contacting our customer support team for further a

In [25]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "answer"],
                    "type": "best_fields",
                }   
            },
        }
    }
}

es_results = es_client.search(index=index_name, body=search_query)
[elem['_score'] for elem in es_results["hits"]["hits"]]

[38.97694, 18.37154, 13.803493, 11.524105, 11.021828]

In [26]:
def elastic_search_hybrid(query, index_name="documents", field='embedding'):
    vector = get_vector(query)
    
    knn_query = {
        "field": "embedding",
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000,
        "boost": 0.5,
    }

    keyword_query = {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "answer"],
                    "type": "best_fields",
                    "boost": 0.5,
                }   
            },
        }
    }

    es_results = es_client.search(
        index=index_name,
        query=keyword_query,
        knn=knn_query,
        size=5
    )

    return [hit["_source"] for hit in es_results["hits"]["hits"]]

In [36]:
result = elastic_search_hybrid(query="How to create an account?")
[elem['question'] for elem in result]

['How can I create an account?',
 'Can I order without creating an account?',
 'How long does shipping take?',
 'How can I contact customer support?',
 'How can I track my order?']

In [38]:
df_ground_truth = pd.read_csv('../data/ground_truth.csv')
df_ground_truth.head()

,question,document
0,How do I make an account on your website?,qDNpG85o
1,Where do I click to sign up for a new account?,qDNpG85o
2,What’s the easiest way to register on your site?,qDNpG85o
3,Can you tell me how to create an account online?,qDNpG85o
4,How do I complete the account registration pro...,qDNpG85o


In [3]:
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo

import os


TZ_INFO = os.getenv("TZ", "America/Chicago")
tz = ZoneInfo(TZ_INFO)


print(f"Script started at {datetime.now(tz)}")
end_time = datetime.now(tz)
start_time = end_time - timedelta(hours=6)
print(f"Generating historical data from {start_time} to {end_time}")

Script started at 2026-08-10 23:12:07.538360-05:00
Generating historical data from 2026-08-10 17:12:07.538541-05:00 to 2026-08-10 23:12:07.538541-05:00
